In [199]:
import pandas as pd

def process_food_data(file_path, counts_file_path):
    foods_df = pd.read_csv(file_path)

    non_zero_tags_columns = [
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_fiber', 'high_fiber', 'low_saturated_fat', 'high_saturated_fat',
        'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium', 'low_calcium', 'high_calcium',
        'low_phosphorus', 'high_phosphorus', 'low_potassium', 'high_potassium', 'low_iron', 'high_iron',
        'low_folic_acid', 'high_folic_acid', 'low_vitamin_c', 'high_vitamin_c', 'low_vitamin_d', 'high_vitamin_d',
        'low_vitamin_b12', 'high_vitamin_b12'
    ]

    non_zero_primary_tags_columns = [
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium'
    ]

    # counting the number of 1's in the tags and primary tags columns
    foods_df['non_zero_tags'] = foods_df[non_zero_tags_columns].sum(axis=1)
    foods_df['non_zero_primary_tags'] = foods_df[non_zero_primary_tags_columns].sum(axis=1)

    foods_df.to_csv(file_path, index=False)

    # these columns (0/1 values) tell us whether a food has a tag (either high or low) corresponding to that macro 
    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]

    # group by macro columns and count the number of rows for each combination
    combination_counts = foods_df.groupby(macro_columns).size().reset_index(name='count')

    # sum of the macro_columns (telling us how many macro tags a food has)
    combination_counts['macro_sum'] = combination_counts[macro_columns].sum(axis=1)

    sorted_combination_counts = combination_counts.sort_values(by='count', ascending=False)

    sorted_combination_counts.to_csv(counts_file_path, index=False)

    return sorted_combination_counts


In [200]:
def count_macro_sum_occurrences(sorted_counts):
    macro_sum_counts = sorted_counts.groupby('macro_sum')['count'].sum().reset_index(name='total_count')
    return macro_sum_counts

#### mixed dishes

In [201]:
file_path = '../processed_data/reduced_mixed_dishes_v4.csv'
counts_file_path = '../processed_data/combination_counts_mixed_dishes.csv'

sorted_counts = process_food_data(file_path, counts_file_path)
display(sorted_counts)

,macro_carb,macro_calorie,macro_sodium,macro_cholesterol,macro_protein,macro_sugar,count,macro_sum
12,1,0,1,1,1,1,199,5
22,1,1,1,1,1,1,96,6
9,1,0,1,0,1,1,84,4
7,1,0,1,0,0,1,51,3
10,1,0,1,1,0,1,49,4
20,1,1,1,1,0,1,44,5
16,1,1,1,0,0,1,41,4
18,1,1,1,0,1,1,25,5
5,1,0,0,1,1,1,17,4
19,1,1,1,1,0,0,10,4


In [202]:
count_macro_sum_occurrences(sorted_counts)

,macro_sum,total_count
0,2,6
1,3,70
2,4,209
3,5,281
4,6,96


Interpretation: 

* 121 foods in the mixed dishes category have 6 primary macro nutrition tags.

### Other foods (not mixed foods)

#### Step 1: join with food_tagging to get the nutrition tags

In [203]:
def process_recommendable_foods(file_path, final_file_path):
    foods_df = pd.read_csv(file_path)
    
    recommendable_df = foods_df[foods_df['recommendable_flag'] == 'recommendable']
    
    recommendable_df = recommendable_df[['food_id', 'food_desc', 'WWEIA_desc', 'ingredient_desc']]
    
    food_tagging_path = '../processed_data/food_tagging.csv'
    food_tagging_df = pd.read_csv(food_tagging_path)
    
    final_df = pd.merge(recommendable_df, food_tagging_df, on='food_id', how='left')
    
    final_df.to_csv(final_file_path, index=False)

In [204]:
def process_foods(file_path, final_file_path):

    foods_df = pd.read_csv(file_path)
    
    foods_df = foods_df[['food_id', 'food_desc', 'WWEIA_desc', 'ingredient_desc']]
    
    food_tagging_path = '../processed_data/food_tagging.csv'
    food_tagging_df = pd.read_csv(food_tagging_path)
    
    final_df = pd.merge(foods_df, food_tagging_df, on='food_id', how='left')
    
    final_df.to_csv(final_file_path, index=False)

In [205]:
file_path = '../processed_data/reduced_meat_seafood_with_recommendations.csv'
final_file_path = '../processed_data/reduced_meat_seafood_final.csv'
process_recommendable_foods(file_path, final_file_path)

In [206]:
file_path = '../processed_data/reduced_plant_protein_with_recommendations.csv'
final_file_path = '../processed_data/reduced_plant_protein_final.csv'
process_recommendable_foods(file_path, final_file_path)

In [207]:
file_path = '../processed_data/reduced_vegetables_potatoes_with_recommendations.csv'
final_file_path = '../processed_data/reduced_vegetables_potatoes_final.csv'
process_recommendable_foods(file_path, final_file_path)

In [208]:
file_path = '../processed_data/reduced_processed_meat.csv'
final_file_path = '../processed_data/reduced_processed_meat_final.csv'
process_foods(file_path, final_file_path)

In [209]:
file_path = '../processed_data/reduced_baked_desserts.csv'
final_file_path = '../processed_data/reduced_baked_desserts_final.csv'
process_foods(file_path, final_file_path)

In [210]:
file_path = '../processed_data/reduced_breads.csv'
final_file_path = '../processed_data/reduced_breads_final.csv'
process_foods(file_path, final_file_path)

#### Step 2: count number of primary macro tags the foods have

##### meat & seafood

In [211]:
file_path = '../processed_data/reduced_meat_seafood_final.csv'
counts_file_path = '../processed_data/combination_counts_meat_seafood.csv'

sorted_counts = process_food_data(file_path, counts_file_path)
display(sorted_counts)

,macro_carb,macro_calorie,macro_sodium,macro_cholesterol,macro_protein,macro_sugar,count,macro_sum
7,1,0,1,1,1,1,103,5
11,1,1,1,1,1,1,65,6
5,1,0,1,1,0,1,7,4
4,1,0,1,0,1,1,5,4
1,1,0,0,1,1,1,4,4
2,1,0,1,0,0,1,2,3
10,1,1,1,1,0,1,2,5
0,1,0,0,1,1,0,1,3
3,1,0,1,0,1,0,1,3
6,1,0,1,1,1,0,1,4


In [212]:
count_macro_sum_occurrences(sorted_counts)

,macro_sum,total_count
0,3,5
1,4,17
2,5,106
3,6,65


##### plant protein

In [213]:
file_path = '../processed_data/reduced_plant_protein_final.csv'
counts_file_path = '../processed_data/combination_counts_plant_protein.csv'

sorted_counts = process_food_data(file_path, counts_file_path)
display(sorted_counts)

,macro_carb,macro_calorie,macro_sodium,macro_cholesterol,macro_protein,macro_sugar,count,macro_sum
5,1,0,1,1,1,1,18,5
4,1,0,1,1,1,0,4,4
8,1,1,1,1,1,1,3,6
3,1,0,1,1,0,1,2,4
0,0,1,1,1,1,1,1,5
1,1,0,0,1,1,1,1,4
2,1,0,1,0,1,1,1,4
6,1,1,1,1,0,1,1,5
7,1,1,1,1,1,0,1,5


In [214]:
count_macro_sum_occurrences(sorted_counts)

,macro_sum,total_count
0,4,8
1,5,21
2,6,3


##### vegetables & potatoes

In [215]:
file_path = '../processed_data/reduced_vegetables_potatoes_final.csv'
counts_file_path = '../processed_data/combination_counts_vegetables_potatoes.csv'

sorted_counts = process_food_data(file_path, counts_file_path)
display(sorted_counts)

,macro_carb,macro_calorie,macro_sodium,macro_cholesterol,macro_protein,macro_sugar,count,macro_sum
5,1,0,1,1,1,1,51,5
11,1,1,1,1,1,1,48,6
6,1,1,0,1,1,1,20,5
2,1,0,0,1,1,1,14,4
4,1,0,1,1,1,0,8,4
10,1,1,1,1,1,0,7,5
1,1,0,0,1,1,0,3,3
3,1,0,1,0,1,1,2,4
0,0,1,1,1,0,1,1,4
7,1,1,1,0,0,0,1,3


In [216]:
count_macro_sum_occurrences(sorted_counts)

,macro_sum,total_count
0,3,4
1,4,25
2,5,80
3,6,48


#####  processed_meat

In [217]:
file_path = '../processed_data/reduced_processed_meat_final.csv'
counts_file_path = '../processed_data/combination_counts_processed_meat.csv'

sorted_counts = process_food_data(file_path, counts_file_path)
display(sorted_counts)

,macro_carb,macro_calorie,macro_sodium,macro_cholesterol,macro_protein,macro_sugar,count,macro_sum
6,1,1,1,1,1,1,30,6
4,1,1,1,1,0,1,18,5
2,1,0,1,1,1,1,12,5
5,1,1,1,1,1,0,3,5
1,1,0,1,1,0,1,2,4
0,1,0,1,0,0,1,1,3
3,1,1,1,0,1,1,1,5


In [218]:
count_macro_sum_occurrences(sorted_counts)

,macro_sum,total_count
0,3,1
1,4,2
2,5,34
3,6,30


##### baked desserts

In [219]:
file_path = '../processed_data/reduced_baked_desserts_final.csv'
counts_file_path = '../processed_data/combination_counts_baked_desserts.csv'

sorted_counts = process_food_data(file_path, counts_file_path)
display(sorted_counts)

,macro_carb,macro_calorie,macro_sodium,macro_cholesterol,macro_protein,macro_sugar,count,macro_sum
20,1,1,1,1,1,1,56,6
19,1,1,1,1,1,0,43,5
6,0,1,1,1,1,1,33,5
15,1,1,0,1,1,1,10,5
16,1,1,1,0,1,0,7,4
17,1,1,1,0,1,1,5,5
5,0,1,1,1,1,0,5,4
8,1,0,0,1,1,0,5,3
2,0,1,1,0,1,0,4,3
9,1,0,0,1,1,1,4,4


In [220]:
count_macro_sum_occurrences(sorted_counts)

,macro_sum,total_count
0,2,1
1,3,12
2,4,26
3,5,94
4,6,56


##### breads

In [221]:
file_path = '../processed_data/reduced_breads_final.csv'
counts_file_path = '../processed_data/combination_counts_breads.csv'
sorted_counts = process_food_data(file_path, counts_file_path)
display(sorted_counts)

,macro_carb,macro_calorie,macro_sodium,macro_cholesterol,macro_protein,macro_sugar,count,macro_sum
13,1,1,1,1,1,1,56,6
12,1,1,1,1,1,0,38,5
11,1,1,1,1,0,1,16,5
6,1,0,1,1,1,1,13,5
10,1,1,1,1,0,0,7,4
5,1,0,1,1,1,0,6,4
8,1,1,1,0,1,0,4,4
1,0,1,1,1,1,1,3,5
9,1,1,1,0,1,1,3,5
0,0,1,1,1,1,0,2,4


In [222]:
count_macro_sum_occurrences(sorted_counts)

,macro_sum,total_count
0,3,1
1,4,21
2,5,74
3,6,56


### Analyzing food_tagging.csv

The goal is to see how many macro tags the foods in the entire list have. We care only about the primary nutrition tags 

In [223]:
def process_macro_counts(food_tagging_path):
    food_tagging_df = pd.read_csv(food_tagging_path)

    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]

    # sum of the macro columns
    food_tagging_df['macro_counts'] = food_tagging_df[macro_columns].sum(axis=1)

    # group by 'macro_counts' and count the number of rows for each unique value
    macro_counts_summary = food_tagging_df.groupby('macro_counts').size().reset_index(name='count')

    macro_counts_summary = macro_counts_summary.sort_values(by='macro_counts', ascending=True)

    return macro_counts_summary

food_tagging_path = '../processed_data/food_tagging.csv'

macro_counts_summary = process_macro_counts(food_tagging_path)

display(macro_counts_summary)


,macro_counts,count
0,0,1
1,2,19
2,3,529
3,4,3020
4,5,4079
5,6,1992


When we consider the primary nutrition tags: `'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol', 'macro_saturated_fat', 'macro_protein', 'macro_sugar'`

Most foods have 4-6 nutrition tags. No foods have 1 nutrition tag.

### Food tags analysis in conjunction with user data

In [224]:
def process_user_info_data(input_file_path, output_file_path):

    columns_to_keep = [
        'SEQN', 'Weight loss/Low calorie diet', 'Low fat/Low cholesterol diet',
        'Low salt/Low sodium diet', 'Sugar free/Low sugar diet', 'Diabetic diet',
        'Weight gain/Muscle building diet', 'Low carbohydrate diet', 'High protein diet',
        'Renal/Kidney diet', 'low_phosphorus', 'low_carb', 'obesity', 'high_calorie',
        'low_calorie', 'hypertension', 'high_potassium', 'low_sodium', 'low_cholesterol',
        'low_saturated_fat', 'low_protein', 'opioid_misuse', 'high_protein', 'diabetes',
        'low_sugar', 'high_fiber', 'high_iron', 'high_folate_acid', 'high_vitamin_b12',
        'high_calcium', 'high_vitamin_c', 'high_vitamin_d'
    ]
    
    df = pd.read_csv(input_file_path)
    
    df_filtered = df[columns_to_keep]
    
    df_filtered.to_csv(output_file_path, index=False)

input_file_path = '../processed_data/user_info_data.csv'
output_file_path = '../processed_data/user_info_data_v2.csv'

process_user_info_data(input_file_path, output_file_path)


#### join with user_tagging to get the macro_ columns for important macros

In [225]:
def merge_user_info_with_tagging(user_info_file, user_tagging_file, output_file):
    user_info_df = pd.read_csv(user_info_file)
    
    user_tagging_df = pd.read_csv(user_tagging_file)
    
    tagging_columns_to_keep = [
        'SEQN', 'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]
    
    user_tagging_df_filtered = user_tagging_df[tagging_columns_to_keep]
    
    merged_df = pd.merge(user_info_df, user_tagging_df_filtered, on='SEQN', how='left')
    
    merged_df.to_csv(output_file, index=False)

user_info_file = '../processed_data/user_info_data_v2.csv'
user_tagging_file = '../processed_data/user_tagging.csv'
output_file = '../processed_data/user_info_data_v3.csv'

merge_user_info_with_tagging(user_info_file, user_tagging_file, output_file)

##### users

In [226]:
def process_user_data(file_path, counts_file_path):
    foods_df = pd.read_csv(file_path)

    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]
    
    # ensure the DataFrame contains only the columns in the macro_columns list
    available_macro_columns = [col for col in macro_columns if col in foods_df.columns]
    
    if not available_macro_columns:
        raise KeyError("None of the specified macro columns are available in the dataframe.")

    # group by the available macro columns and count the number of rows for each combination
    combination_counts = foods_df.groupby(available_macro_columns).size().reset_index(name='count')

    # sum of the macro columns in each row
    combination_counts['macro_sum'] = combination_counts[available_macro_columns].sum(axis=1)

    sorted_combination_counts = combination_counts.sort_values(by='count', ascending=False)

    sorted_combination_counts.to_csv(counts_file_path, index=False)

    return sorted_combination_counts

file_path = '../processed_data/user_info_data_v3.csv'
counts_file_path = '../processed_data/combination_counts_users.csv'

sorted_counts = process_user_data(file_path, counts_file_path)

display(sorted_counts)


,macro_carb,macro_calorie,macro_sodium,macro_cholesterol,macro_protein,macro_sugar,count,macro_sum
0,0,0,0,0,0,0,36179,0
16,0,1,0,0,0,0,29270,1
24,0,1,1,0,0,0,7842,2
8,0,0,1,0,0,0,5680,1
20,0,1,0,1,0,0,2111,2
4,0,0,0,1,0,0,1903,1
25,0,1,1,0,0,1,1767,3
17,0,1,0,0,0,1,1713,2
28,0,1,1,1,0,0,1519,3
18,0,1,0,0,1,0,1195,2


In [227]:
count_macro_sum_occurrences(sorted_counts)

,macro_sum,total_count
0,0,36179
1,1,39014
2,2,16015
3,3,6019
4,4,1143
5,5,78


In [228]:
def process_user_info(user_info_file):
    user_info_df = pd.read_csv(user_info_file)

    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]

    user_info_df['sum_macro'] = user_info_df[macro_columns].sum(axis=1)

    filtered_df = user_info_df[user_info_df['sum_macro'] == 1]

    macro_counts = {
        'single_macro': macro_columns,
        'count_users': [filtered_df[macro].sum() for macro in macro_columns]
    }

    result_df = pd.DataFrame(macro_counts)

    return result_df

user_info_file = '../processed_data/user_info_data_v3.csv'

result_df = process_user_info(user_info_file)

display(result_df)


,single_macro,count_users
0,macro_carb,68
1,macro_calorie,29270
2,macro_sodium,5680
3,macro_cholesterol,1903
4,macro_protein,995
5,macro_sugar,1098


## I. Users with single condition

### 1. Obesity

In [160]:
# identify most common diets among these users with obesity

def process_user_diet_counts(user_info_file):
    user_info_df = pd.read_csv(user_info_file)

    filtered_df = user_info_df[
        (user_info_df['obesity'] == 1) &
        (user_info_df['hypertension'] == 0) &
        (user_info_df['opioid_misuse'] == 0) &
        (user_info_df['diabetes'] == 0)
    ]

    diet_columns = [
        'Weight loss/Low calorie diet', 'Low fat/Low cholesterol diet', 'Low salt/Low sodium diet',
        'Sugar free/Low sugar diet', 'Diabetic diet', 'Weight gain/Muscle building diet',
        'Low carbohydrate diet', 'High protein diet', 'Renal/Kidney diet'
    ]

    diet_counts = {
        'diet': diet_columns,
        'count_seqn': [filtered_df[col].sum() for col in diet_columns]
    }

    result_df = pd.DataFrame(diet_counts)

    total_users = len(filtered_df)

    result_df['percentage'] = (result_df['count_seqn'] / total_users) * 100

    result_df = result_df.sort_values(by='count_seqn', ascending=False)

    return result_df

user_info_file = '../processed_data/user_info_data_v3.csv'

diet_counts_df = process_user_diet_counts(user_info_file)

display(diet_counts_df)


,diet,count_seqn,percentage
0,Weight loss/Low calorie diet,1598,12.797309
1,Low fat/Low cholesterol diet,281,2.250340
4,Diabetic diet,266,2.130215
2,Low salt/Low sodium diet,212,1.697766
6,Low carbohydrate diet,156,1.249299
3,Sugar free/Low sugar diet,90,0.720750
7,High protein diet,41,0.328341
8,Renal/Kidney diet,12,0.096100
5,Weight gain/Muscle building diet,3,0.024025


The percentages in the table do not sum to 100% because the conditions we are counting (e.g., Weight loss/Low calorie diet == 1, Low fat/Low cholesterol diet == 1, etc.) are not mutually exclusive. This means that users can belong to more than one of these categories (for example, a user could be following both a Weight loss/Low calorie diet and a Low fat/Low cholesterol diet at the same time).

Since users can be counted in more than one category, the total of the percentages can exceed or fall short of 100%, depending on how many users follow multiple diets.

#### 1.1. Easy questions

We observe that the most common diet for these users is Weight loss/low calorie diet, which suggests that to construct easy questions, we can just identify foods with high_calorie/low_calorie tags.

Step 1: identify users in this category: main filtering criteria: 

```
(user_info_df['obesity'] == 1) & 
(user_info_df['Weight loss/Low calorie diet'] == 1) &
(user_info_df['sum_macro'] == 1) &
(user_info_df['macro_calorie'] == 1) 
```

In [187]:
def process_user_info(user_info_file):
    user_info_df = pd.read_csv(user_info_file)

    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]

    # sum of the macro columns
    user_info_df['sum_macro'] = user_info_df[macro_columns].sum(axis=1)

    filtered_df = user_info_df[
        (user_info_df['obesity'] == 1) & 
        (user_info_df['Weight loss/Low calorie diet'] == 1) &
        (user_info_df['sum_macro'] == 1) &
        (user_info_df['macro_calorie'] == 1) 
    ]

    return filtered_df

user_info_file = '../processed_data/user_info_data_v3.csv'

filtered_users_df = process_user_info(user_info_file)

In [188]:
num_users = len(filtered_users_df)
print(f"Number of users in filtered_users_df: {num_users}")

Number of users in filtered_users_df: 855


For these users, we can just identify foods that have either low_calorie or high_calorie tags (these foods can have other tags, but we don't care for now since these are easy questions, which are questions with a one-on-one match or contradict)

In [155]:
file_path = '../processed_data/reduced_mixed_dishes_v4.csv'
foods_df = pd.read_csv(file_path)
filtered_foods_df = foods_df[foods_df['macro_calorie'] == 1]
num_foods = len(filtered_foods_df)
print(f"Number of foods in filtered_foods_df: {num_foods}")

Number of foods in filtered_foods_df: 240


So for the **855** users with a single condition that is "Obesity", who are under the "Weight loss/Low calorie diet", and with only one macro tag in the calorie category (either high_calorie or low_calorie), we can identify **240** mixed dishes that can be used to create easy questions for these users.

If we want to add foods from other categories, we can also do the same filtering. For example, from the meat/seafood category, we can identify 69 more foods that can be used to create easy questions for this group of users

In [105]:
file_path = '../processed_data/reduced_meat_seafood_final.csv'
foods_df = pd.read_csv(file_path)
filtered_foods_df = foods_df[foods_df['macro_calorie'] == 1]
num_foods = len(filtered_foods_df)
print(f"Number of foods in filtered_foods_df: {num_foods}")

Number of foods in filtered_foods_df: 69


Or if we want to add foods from the vegetables/potatoes group, we can add 79 more foods

In [120]:
file_path = '../processed_data/reduced_vegetables_potatoes_final.csv'
foods_df = pd.read_csv(file_path)
filtered_foods_df = foods_df[foods_df['macro_calorie'] == 1]
num_foods = len(filtered_foods_df)
print(f"Number of foods in filtered_foods_df: {num_foods}")

Number of foods in filtered_foods_df: 79


#### 1.2. Medium/hard questions

To construct medium/hard questions, the prerequisite is we filter for users with multiple macro tags (in the important categories only: carb, calorie, sodium, cholesterol, protein, sugar), and foods with multiple macro tags also in these important categories only. 

To reduce the number of users that we can potentially use to construct medium/hard questions, we might need to only look at the top most common diets for people with obesity: "Weight loss/Low calorie diet", "Low fat/Low cholesterol diet", "Diabetic diet". 

For the first two diets, it's clear that we need to care about the calorie and cholesterol tags. What about diabetic diet? This diet involves multiple nutrition tags. What are these tags?

In [170]:
def count_macro_columns_for_diabetic_diet(user_info_file):
    user_info_df = pd.read_csv(user_info_file)

    filtered_df = user_info_df[user_info_df['Diabetic diet'] == 1]

    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]

    macro_counts = {
        'macro': macro_columns,
        'count_seqn': [filtered_df[macro].sum() for macro in macro_columns]
    }

    result_df = pd.DataFrame(macro_counts)

    result_df = result_df.sort_values(by='count_seqn', ascending=False)

    return result_df

user_info_file = '../processed_data/user_info_data_v3.csv'

macro_counts_df = count_macro_columns_for_diabetic_diet(user_info_file)

display(macro_counts_df)

,macro,count_seqn
5,macro_sugar,1529
1,macro_calorie,1097
2,macro_sodium,841
4,macro_protein,445
3,macro_cholesterol,141
0,macro_carb,29


So it seems like for people who are on a "Diabetic diet", sugar and calorie are the 2 most important macros to consider. 

Together with the two other diets, we can potentially look at foods that have more than 1 of these macro tags: calorie, cholesterol, sugar, for this group of users (users who have only one health condition, that is Obesity)

First, let's filter for these users: 

```
(user_info_df['obesity'] == 1) & 
(user_info_df['sum_macro'] > 1) & 
((user_info_df['Weight loss/Low calorie diet'] == 1) | (user_info_df['Low fat/Low cholesterol diet'] == 1) | (user_info_df['Diabetic diet'] == 1)) 
```

In [193]:
def process_user_info(user_info_file):
    user_info_df = pd.read_csv(user_info_file)

    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]

    # sum of the macro columns
    user_info_df['sum_macro'] = user_info_df[macro_columns].sum(axis=1)

    important_macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sugar'
    ]

    # sum of the important_macro columns
    user_info_df['sum_important_macro'] = user_info_df[important_macro_columns].sum(axis=1)

    filtered_df = user_info_df[
        (user_info_df['obesity'] == 1) & 
        (user_info_df['sum_macro'] > 1) & 
        (user_info_df['sum_important_macro'] >= 2) & 
        ((user_info_df['Weight loss/Low calorie diet'] == 1) | (user_info_df['Low fat/Low cholesterol diet'] == 1) | (user_info_df['Diabetic diet'] == 1)) 
    ]

    return filtered_df

user_info_file = '../processed_data/user_info_data_v3.csv'

filtered_users_df = process_user_info(user_info_file)

In [194]:
num_users = len(filtered_users_df)
print(f"Number of users in filtered_users_df: {num_users}")

Number of users in filtered_users_df: 1178


Next, identify foods with more than one tag in these macros: calorie, cholesterol, sugar

In [198]:
def process_dishes(file_path):
    df = pd.read_csv(file_path)

    # fefine the columns to sum
    selected_macro_columns = [
        'macro_calorie', 'macro_cholesterol', 'macro_sugar'
    ]

    df['sum_selected_macros'] = df[selected_macro_columns].sum(axis=1)

    # filter the rows where sum_selected_macros >= 2
    filtered_df = df[df['sum_selected_macros'] >= 2]

    return filtered_df

file_path = '../processed_data/reduced_mixed_dishes_v4.csv'

filtered_dishes = process_dishes(file_path)

print(f"Number of foods in filtered_foods_df: {len(filtered_dishes)}")

Number of foods in filtered_foods_df: 496


So for the **1178** users with a single condition that is "Obesity", who are under the "Weight loss/Low calorie diet", or "Low fat/Low cholesterol diet", or "Diabetic diet", and with more than one important macro tags and more than two tags in these categories: calorie, cholesterol, sugar, we can identify **496** mixed dishes that can be used to create easy questions for these users

From this list of 1178 users and 469 foods, we can identify user-food pairs that fall under each category of medium or hard: 

* medium: multi-on-multi match or contradict 
* hard: mixed combination of match or contradict

And same as the easy questions case, we can also add foods from other categories if needed. For example, we can add 182 foods from the meat/seafood category

In [197]:
file_path = '../processed_data/reduced_meat_seafood_final.csv'
filtered_dishes = process_dishes(file_path)
print(f"Number of foods in filtered_foods_df: {len(filtered_dishes)}")

Number of foods in filtered_foods_df: 182


### 2. Hypertension

For people with only one condition that is hypertension, we can follow the same approach as above to identify users and foods that can be used to construct easy questions, medium/hard questions

### 3. Opioid Misuse

Can be done similar to previous case

### 4. Diabetes

Can be done similar to previous case

# Users with multiple conditions

We have to make a statement that questions constructed based on users with more than one condition are almost always going to be in the medium and hard categories since they involve multiple diet tags.

## II. Users with two conditions

Overlaps of 2 statuses:

* obesity and hypertension: 3974 users
* obesity and opioid_misuse: 1107 users
* obesity and diabetes: 2045 users
* hypertension and opioid_misuse: 601 users
* hypertension and diabetes: 1178 users
* opioid_misuse and diabetes: 289 users

The combination with most number of users is obesity & hypertension with 3974 users. Let's look at this group of users first. 

In [233]:
def count_macro_columns_for_diabetic_diet(user_info_file):

    user_info_df = pd.read_csv(user_info_file)

    # filter rows where 'Diabetic diet' == 1, 'obesity' == 1, and 'hypertension' == 1
    filtered_df = user_info_df[
        (user_info_df['obesity'] == 1) &
        (user_info_df['hypertension'] == 1) & 
        (user_info_df['opioid_misuse'] == 0) & 
        (user_info_df['diabetes'] == 0)
    ]

    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]

    macro_counts = {
        'macro': macro_columns,
        'count_seqn': [filtered_df[macro].sum() for macro in macro_columns]
    }

    result_df = pd.DataFrame(macro_counts)

    result_df = result_df.sort_values(by='count_seqn', ascending=False)

    return result_df

user_info_file = '../processed_data/user_info_data_v3.csv'

macro_counts_df = count_macro_columns_for_diabetic_diet(user_info_file)

display(macro_counts_df)


,macro,count_seqn
1,macro_calorie,3250
2,macro_sodium,3250
3,macro_cholesterol,509
4,macro_protein,433
5,macro_sugar,124
0,macro_carb,36


So it seems like for users with obesity and hypertension, calorie and sodium are the most important tags (which makes sense). With this in mind, we can filter for foods with these two tags (they might have other tags, which we don't really care about in this case)

In [251]:
def process_dishes(file_path):
    df = pd.read_csv(file_path)

    selected_macro_columns = [
        'macro_calorie', 'macro_sodium'
    ]

    # sum of the selected columns
    df['sum_selected_macros'] = df[selected_macro_columns].sum(axis=1)

    # filter the rows where sum_selected_macros == 2, i.e. the food has the 2 macro tags of our interest
    filtered_df = df[df['sum_selected_macros'] == 2]

    return filtered_df

file_path = '../processed_data/reduced_mixed_dishes_v4.csv'

filtered_dishes = process_dishes(file_path)

print(f"Number of foods in filtered_foods_df: {len(filtered_dishes)}")

Number of foods in filtered_foods_df: 235


So for the ~3k users with both obesity and hypertension, we identify 235 mixed dishes that can potentially be used to construct medium/hard questions

NOTE: here, I get 3160 users

In [247]:
user_info_file = '../processed_data/user_info_data.csv'

user_info_df = pd.read_csv(user_info_file)

filtered_df = user_info_df[
    (user_info_df['obesity'] == 1) &
    (user_info_df['hypertension'] == 1) & 
    (user_info_df['opioid_misuse'] == 0) & 
    (user_info_df['diabetes'] == 0)
]

print(len(filtered_df))

3160


But seems like Jason got 3974 users because of this logic, where we don't specify that opioid_misuse == 0 and diabetes == 0

In [248]:
user_info_file = '../processed_data/user_info_data.csv'

user_info_df = pd.read_csv(user_info_file)

filtered_df = user_info_df[
    (user_info_df['obesity'] == 1) &
    (user_info_df['hypertension'] == 1)
]

print(len(filtered_df))

3974


Of course, we can extend this analysis to other combinations of two conditions, besides obesity & hypertension

## III. Users with three conditions

Overlaps of 3 statuses:

* obesity, hypertension, and opioid_misuse: 245 users
* obesity, hypertension, and diabetes: 616 users
* obesity, opioid_misuse, and diabetes: 174 users
* hypertension, opioid_misuse, and diabetes: 84 users

The most common combination is obesity & hypertension & diabetes.

In [242]:
def count_macro_columns_for_diabetic_diet(user_info_file):
    user_info_df = pd.read_csv(user_info_file)

    # filter rows where 'obesity' == 1, 'hypertension' == 1, and 'hypertension' == 1
    filtered_df = user_info_df[
        (user_info_df['obesity'] == 1) &
        (user_info_df['hypertension'] == 1) & 
        (user_info_df['opioid_misuse'] == 0) & 
        (user_info_df['diabetes'] == 1)
    ]

    print(len(filtered_df))

    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]

    # count the number of SEQN for each macro column where the value is 1
    macro_counts = {
        'macro': macro_columns,
        'count_seqn': [filtered_df[macro].sum() for macro in macro_columns]
    }

    result_df = pd.DataFrame(macro_counts)

    result_df = result_df.sort_values(by='count_seqn', ascending=False)

    return result_df

user_info_file = '../processed_data/user_info_data_v3.csv'

macro_counts_df = count_macro_columns_for_diabetic_diet(user_info_file)

display(macro_counts_df)


581


,macro,count_seqn
1,macro_calorie,581
2,macro_sodium,581
5,macro_sugar,581
4,macro_protein,144
3,macro_cholesterol,86
0,macro_carb,10


So it seems like for users with obesity & hypertension & diabetes, calorie, sodium and sugar are the most important tags (which makes sense). With this in mind, we can filter for foods with these three tags (they might have other tags, which we don't really care about in this case)

In [252]:
def process_dishes(file_path):
    df = pd.read_csv(file_path)

    selected_macro_columns = [
        'macro_calorie', 'macro_sodium', 'macro_sugar'
    ]

    # sum of the selected columns
    df['sum_selected_macros'] = df[selected_macro_columns].sum(axis=1)

    # filter the rows where sum_selected_macros == 3, i.e. the food has all 3 macro tags of our interest
    filtered_df = df[df['sum_selected_macros'] == 3]

    return filtered_df

file_path = '../processed_data/reduced_mixed_dishes_v4.csv'

filtered_dishes = process_dishes(file_path)

print(f"Number of foods in filtered_foods_df: {len(filtered_dishes)}")

Number of foods in filtered_foods_df: 207


So for the ~500 users with obesity & hypertension & diabetes, we identify 207 mixed dishes that can potentially be used to construct medium/hard questions

NOTE: here, I get 569 users

In [253]:
user_info_file = '../processed_data/user_info_data.csv'

user_info_df = pd.read_csv(user_info_file)

filtered_df = user_info_df[
    (user_info_df['obesity'] == 1) &
    (user_info_df['hypertension'] == 1) & 
    (user_info_df['opioid_misuse'] == 0) & 
    (user_info_df['diabetes'] == 1)
]

print(len(filtered_df))

569


But seems like Jason got 616 users because of this logic, where we don't specify that opioid_misuse == 0

In [254]:
filtered_df = user_info_df[
    (user_info_df['obesity'] == 1) &
    (user_info_df['hypertension'] == 1) & 
    (user_info_df['diabetes'] == 1)
]

print(len(filtered_df))

616


## IV. Users with three conditions

Overlap of all 4 statuses:
* Overlap between all four statuses: 47 users

In [255]:
def count_macro_columns_for_diabetic_diet(user_info_file):
    user_info_df = pd.read_csv(user_info_file)

    filtered_df = user_info_df[
        (user_info_df['obesity'] == 1) &
        (user_info_df['hypertension'] == 1) & 
        (user_info_df['opioid_misuse'] == 1) & 
        (user_info_df['diabetes'] == 1)
    ]

    print(len(filtered_df))

    macro_columns = [
        'macro_carb', 'macro_calorie', 'macro_sodium', 'macro_cholesterol',
        'macro_protein', 'macro_sugar'
    ]

    # count the number of SEQN for each macro column where the value is 1
    macro_counts = {
        'macro': macro_columns,
        'count_seqn': [filtered_df[macro].sum() for macro in macro_columns]
    }

    result_df = pd.DataFrame(macro_counts)
    result_df = result_df.sort_values(by='count_seqn', ascending=False)

    return result_df

user_info_file = '../processed_data/user_info_data_v3.csv'

macro_counts_df = count_macro_columns_for_diabetic_diet(user_info_file)

display(macro_counts_df)


53


,macro,count_seqn
1,macro_calorie,53
2,macro_sodium,53
5,macro_sugar,53
4,macro_protein,17
3,macro_cholesterol,8
0,macro_carb,0


So it seems like for users with all four conditions, calorie, sodium, sugar and protein are the most important tags. With this in mind, we can filter for foods with these four tags (they might have other tags, which we don't really care about in this case)

In [256]:
def process_dishes(file_path):
    df = pd.read_csv(file_path)

    selected_macro_columns = [
        'macro_calorie', 'macro_sodium', 'macro_sugar', 'macro_protein'
    ]

    # sum of the selected columns
    df['sum_selected_macros'] = df[selected_macro_columns].sum(axis=1)

    # filter the rows where sum_selected_macros == 4, which means the food has all 4 tags
    filtered_df = df[df['sum_selected_macros'] == 4]

    return filtered_df

file_path = '../processed_data/reduced_mixed_dishes_v4.csv'

filtered_dishes = process_dishes(file_path)

print(f"Number of foods in filtered_foods_df: {len(filtered_dishes)}")

Number of foods in filtered_foods_df: 122


So for users with all four conditions, we identify 122 mixed dishes that can potentially be used to construct medium/hard questions